<a href="https://colab.research.google.com/github/kasyap22307/24B11AI134_DriveReady_OPC_2026/blob/main/Day2/MyBusinessBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 2: Satish V Mix Photo Studio & Xerox chatbot
Prompts, memory, dicts/JSON, functions, decorators and try/except.

In [ ]:
%pip install -q openai

In [ ]:
import os, json, math, functools
from openai import OpenAI, RateLimitError, APIConnectionError, AuthenticationError

# ---------- Safety net: build `client` if the Day 1 cells weren't run ----------
try:
    client, MODEL
except NameError:
    try:
        from google.colab import userdata
        os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    except ImportError:
        from getpass import getpass
        os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")
    client = OpenAI(api_key=os.getenv("GROQ_API_KEY"),
                    base_url="https://api.groq.com/openai/v1")
    MODEL = "openai/gpt-oss-20b"

## Business details and menu

In [ ]:
STUDIO_NAME = "Satish V Mix"
OWNER_NAME  = "Satish"
OWNER_PHONE = "9876543210"

# Task 5 - the menu as a dict: service -> price in rupees
MENU = {
    "passport photos to capture the image of customer (set of 8)": 100,  # 8 copies, ready in 15 minutes
    "passport photos (if photo is available with the customer)": 80,
    "passport photos (if photo already taken in the owner shop and the date captured is less than 2.5 years )": 60,
    "xerox black & white (per page)": 2,
    "xerox colour (per page)": 10,
    "printing from pen drive/WhatsApp (per page)": 5,
    "lamination A4": 40,
    "lamination Aadhaar card": 30,
    "lamination above A4 size": 60,
    "big size Aadhaar card": 100,
    "PVC card printing": 100,
    "4x6 photo paper printing(per page)": 30,
}

## System prompt: Role, Facts, Rules, Format

In [ ]:
SYSTEM = f"""ROLE:   You are the assistant for {STUDIO_NAME} Photo Studio & Xerox.
FACTS:  Prices in rupees: {json.dumps(MENU)}
        Passport size photos (set of 8 copies): Rs 100 if we take the photo (ready in 15 minutes),
        Rs 80 if the customer already has the photo, Rs 60 if the photo was already taken at our shop
        and was captured less than 2.5 years ago. Walk-ins welcome.
        4x6 photo paper printing Rs 30 per page.
        Lamination: A4 Rs 40, Aadhaar card Rs 30, any size above A4 Rs 60.
        Big size Aadhaar card Rs 100. PVC card printing Rs 100.
        Open 9 AM to 8 PM, Monday to Saturday. Closed on Sundays.
        Owner contact for wedding, event, photography and video enquiries: {OWNER_NAME}, {OWNER_PHONE}.
RULES:  Use ONLY the FACTS. Never invent prices, names or phone numbers.
        Passport photos are always given in multiples of 8 (a 4x6 photo paper prints 8 photos),
        never fewer than 8. If a customer asks for fewer, explain this and quote the price for one set of 8.
        If the customer doesn't say which passport photo case applies, quote Rs 100 (we take the photo)
        and mention the Rs 80 option (they have the photo) and the Rs 60 option (photo taken at our shop
        less than 2.5 years ago). Never offer Rs 60 for photos taken more than 2.5 years ago.
        For ANY question about wedding, event, album, photography shoot or video shoot/editing:
        do NOT give any price, package or delivery time.
        Reply only: "For wedding and video enquiries, please contact the owner: {OWNER_NAME}, {OWNER_PHONE}. Wedding bookings must be informed at least 2 days in advance."
        If the answer isn't in FACTS, say exactly: "I'll check with the owner." and give the owner's contact.
        Never promise a delivery time except the 15 minutes for passport photos.
        Remember the customer's name and order during the chat.
        Reply in the same language as the customer's message (Telugu question -> Telugu reply).
FORMAT: Under 40 words, friendly."""

## Decorator and `get_price()`

In [ ]:
def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"   [log] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper

@log_call
def get_price(service: str, qty: int = 1) -> int:
    """Return the total price in rupees. For passport photos, qty is number of photos
    (rounded UP to sets of 8). For xerox/printing, qty is number of pages.
    For lamination, big Aadhaar card, PVC card and 4x6 photo printing, qty is number of items/pages.
    Passport options: "passport photos" (we take the photo, Rs 100/set), "passport photos own photo"
    (Rs 80/set), "passport photos old photo" (taken at our shop <2.5 years ago, Rs 60/set).
    Returns -1 if the service is not on the menu."""
    key = service.lower().strip()
    # passport photo sets of 8: new photo Rs 100 / customer has photo Rs 80 / our old photo (<2.5 yrs) Rs 60
    passport = {"passport": 100, "passport photos": 100, "passport photo": 100,
                "passport photos own photo": 80, "passport photo own photo": 80,
                "passport photos old photo": 60, "passport photo old photo": 60}
    if key in passport:
        return math.ceil(qty / 8) * passport[key]
    fixed = {"lamination a4": 40, "lamination aadhaar": 30, "lamination aadhar": 30,
             "lamination above a4": 60, "big aadhaar card": 100, "big aadhar card": 100,
             "pvc card": 100, "pvc card printing": 100, "4x6 photo printing": 30,
             "4x6 photo": 30, "4x6 print": 30}
    if key in fixed:
        return fixed[key] * qty
    price_per_page = {"xerox": 2, "xerox bw": 2, "xerox colour": 10,
                      "xerox color": 10, "printing": 5}
    if key in price_per_page:
        return price_per_page[key] * qty
    return -1   # not on the menu: no crash

## Safe API call (try/except)

In [ ]:
# ---------- One safe API call used by everything (try/except) ----------
def ask_model(messages: list, temperature: float = 0.2) -> str:
    try:
        r = client.chat.completions.create(model=MODEL, messages=messages,
                                           temperature=temperature)
        return r.choices[0].message.content
    except RateLimitError:
        return "We're a little busy. Please try again in a minute."
    except AuthenticationError:
        return "Setup problem: please tell the owner the API key isn't working."
    except APIConnectionError:
        return "I can't reach the server right now. Please try again shortly."

## Memory by hand


In [ ]:
# ---------- Memory by hand (Task 4: set MAX_TURNS = 2 to test) ----------
history = [{"role": "system", "content": SYSTEM}]
MAX_TURNS = 8   # messages kept besides the system message

def chat(user_text: str) -> str:
    history.append({"role": "user", "content": user_text})
    recent = [history[0]] + history[1:][-MAX_TURNS:]     # system + last few
    reply = ask_model(recent)
    history.append({"role": "assistant", "content": reply})
    return reply

## Task 2: few-shot classifier

In [ ]:
# ---------- Task 2: few-shot classifier ----------
LABELS = "order, question, wedding_enquiry or complaint"
EXAMPLES = [
    ("need 20 colour xerox copies",           "order"),
    ("what time do you open?",                "question"),
    ("how much for a wedding photo shoot?",   "wedding_enquiry"),
    ("my passport photos came out blurry",    "complaint"),
]

def classify(message: str) -> str:
    msgs = [{"role": "system",
             "content": f"Label the customer message with one word: {LABELS}."}]
    for text, label in EXAMPLES:
        msgs.append({"role": "user", "content": text})
        msgs.append({"role": "assistant", "content": label})
    msgs.append({"role": "user", "content": message})
    return ask_model(msgs, temperature=0).strip().lower()

## Task 3: order to JSON to dict

In [ ]:
# ---------- Task 3: order -> JSON -> dict ----------
ORDER_SYSTEM = """Extract the order from the customer's message.
Return ONLY JSON, no other text, with these keys:
name (text), service (text: passport photos, xerox black & white, xerox colour, printing, lamination A4, lamination Aadhaar, lamination above A4, big Aadhaar card, PVC card or 4x6 photo printing),
qty (number), urgent (true/false)."""

def extract_order(message: str):
    reply = ask_model([{"role": "system", "content": ORDER_SYSTEM},
                       {"role": "user", "content": message}], temperature=0)
    clean = reply.replace("```json", "").replace("```", "").strip()   # remove fences
    try:
        return json.loads(clean)
    except json.JSONDecodeError:
        print("   [warn] model did not return valid JSON:", reply)
        return None

## Demos: prices, rules test, classifier, JSON


In [ ]:
print("=== Price function ===")
get_price("xerox", 50)             # 100
get_price("passport photos", 3)    # 100 (rounded up to 8)
get_price("passport photos", 10)   # 200 (2 sets)
get_price("lamination a4", 2)      # 80
get_price("pvc card")              # 100
get_price("wedding")               # -1 (not on menu)

print("\n=== Task 1: remove RULES, then ask this ===")
print("(Delete the RULES block in SYSTEM, run again, and compare these answers)")
for q in ["How much for a wedding shoot?", "Can I get 3 passport photos?"]:
    print(f"Q: {q}\nA: {chat(q)}\n")
history[:] = history[:1]   # reset memory after the demo

print("=== Task 2: classifier on new messages ===")
tests = ["I need 5 pages laminated",
         "do you work on sundays?",
         "can you shoot my sister's engagement?",
         "you charged me Rs 10 for black and white",
         "I want 16 passport photos right now"]
for t in tests:
    print(f"{t!r:45} -> {classify(t)}")

print("\n=== Task 3: JSON order ===")
order = extract_order("hi Ravi here, need 16 passport photos, urgent pls")
print(order)
if order:
    print("Service:", order.get("service"), "| Qty:", order.get("qty"))
    if order.get("service", "").lower().startswith("passport"):
        print("Price:", get_price("passport photos", int(order["qty"])))

## Live chat
Type `bye`, `exit` or `quit` to stop. Commands: `/order <text>`, `/classify <text>`, `/history`.

In [ ]:
print(f"=== Chat with {STUDIO_NAME} Photo Studio & Xerox ===")
while True:
    text = input("You: ").strip()
    if text.lower() in ("bye", "exit", "quit"):
        print(f"Bot: Thank you for visiting {STUDIO_NAME}!")
        break
    elif text.startswith("/order "):
        print(extract_order(text[7:]))
    elif text.startswith("/classify "):
        print(classify(text[10:]))
    elif text == "/history":
        print(f"{len(history)} messages in history (MAX_TURNS={MAX_TURNS})")
    else:
        print("Bot:", chat(text))